In [1]:
%pip install pandas pyarrow scikit-learn numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


# Обработка Boarding

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope

def spies_report(X, ID, top_k=10, contamination=0.01):
    if isinstance(X, pd.DataFrame):
        X_vals = X.values
    else:
        X_vals = X
        
    if isinstance(ID, pd.Series):
        ids_arr = ID.values
    else:
        ids_arr = np.array(ID)
        
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_vals)
    
    models = {}
    
    print("Поиск аномалий")
    
    if_model = IsolationForest(n_estimators=150, contamination=contamination, random_state=42, n_jobs=-1)
    if_model.fit(X_vals)
    if_scores = if_model.decision_function(X_vals)
    models['IsolationForest'] = -if_scores
    
    lof_model = LocalOutlierFactor(n_neighbors=20, contamination=contamination, n_jobs=-1)
    lof_model.fit(X_scaled)
    models['LOF'] = -lof_model.negative_outlier_factor_
    
    svm_model = OneClassSVM(nu=contamination, kernel='rbf', gamma='scale')
    svm_model.fit(X_scaled)
    svm_scores = svm_model.decision_function(X_scaled)
    models['OneClassSVM'] = -svm_scores
    
    try:
        ee_model = EllipticEnvelope(contamination=contamination, random_state=42)
        ee_model.fit(X_scaled)
        ee_scores = ee_model.decision_function(X_scaled)
        models['EllipticEnvelope'] = -ee_scores
    except Exception as e:
        print(f"EllipticEnvelope пропущен {e}")
        
    nn_model = NearestNeighbors(n_neighbors=20, n_jobs=-1)
    nn_model.fit(X_scaled)
    distances, _ = nn_model.kneighbors(X_scaled)
    models['KNN_Distance'] = np.mean(distances, axis=1)

    rank_records = []
    
    print("\n" + "-" * 70)
    print("Подозрительные")
    print("-" * 70)
    
    for name, scores in models.items():
        df_m = pd.DataFrame({
            'ID': ids_arr,
            'Anomaly_Score': scores
        }).sort_values(by='Anomaly_Score', ascending=False)
        
        top_df = df_m.head(top_k).copy()
        top_df['Rank'] = range(1, len(top_df) + 1)
        top_df['Model'] = name
        rank_records.append(top_df)
        
        print(f"\n--- {name} ---")
        print(top_df[['ID', 'Anomaly_Score', 'Rank']].to_string(index=False))
        print("-" * 50)
        
    all_tops = pd.concat(rank_records)
    
    all_tops['Points'] = (top_k - all_tops['Rank'] + 1)
    
    summary = all_tops.groupby('ID').agg(
        Total_Score=('Points', 'sum'),
        Models_Count=('Model', 'count'),
        Models_List=('Model', lambda x: ", ".join(x))
    ).reset_index().sort_values(by=['Total_Score', 'Models_Count'], ascending=False)
    
    return summary

In [4]:
BoardingDF = pd.read_csv("../resources/Boarding.csv", sep="|", engine="pyarrow")

aggregation_rules = {
    'PassengerFirstName': 'mean',
    'PassengerSecondName': 'mean',
    'PassengerLastName': 'mean',
    'PassengerSex': 'first',
    'PassengerAge': 'first',
    'PassportSeriesLeft': 'first',
    'PassportSeriesRight': 'first',
    
    'TicketNumber': 'count',
    'FlightTime': 'mean',
    'FlightNumber': 'mean',
    'CodeShare': 'mean',
    'Destination': 'max',
    
    'boardFlightMatch': 'mean',
    'OperatFlightMatch': 'mean',
    'PassBoardingMatch': 'mean',
    'SirenaBoardingMatch': 'mean',
    
    'BaggageTransit': 'sum',
    'BaggageDelayed': 'sum',
    'BaggageRegistered': 'sum',
    
    'Flight_DayOfWeek': 'mean',
    'Flight_Month': 'nunique'
}

passenger_df = BoardingDF.groupby('Document').agg(aggregation_rules).reset_index()

IDS = passenger_df['Document'].copy()
X = passenger_df.drop(columns=['Document'])

spies_report(X, IDS)

Поиск аномалий


/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(



----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
         ID  Anomaly_Score  Rank
5293 500602       0.101237     1
0977 338850       0.096909     2
1311 440150       0.086798     3
8176 892394       0.086109     4
1181 213093       0.083668     5
1379 543723       0.083184     6
1548 113497       0.079667     7
0185 106233       0.078386     8
6937 933652       0.077325     9
2294 770067       0.076448    10
--------------------------------------------------

--- LOF ---
         ID  Anomaly_Score  Rank
2420 116077       1.637272     1
1028 207088       1.618745     2
8537 137481       1.606997     3
5405 898615       1.595745     4
7544 655717       1.585977     5
0977 338850       1.579689     6
1530 068060       1.578621     7
1810 413944       1.577679     8
4076 926383       1.566683     9
7314 180345       1.564469    10
---------------------------

,ID,Total_Score,Models_Count,Models_List
2,0977 338850,34,4,"IsolationForest, LOF, OneClassSVM, KNN_Distance"
21,5293 500602,28,3,"IsolationForest, OneClassSVM, KNN_Distance"
20,4992 905133,18,3,"OneClassSVM, EllipticEnvelope, KNN_Distance"
3,1028 207088,15,2,"LOF, KNN_Distance"
15,3050 260754,15,2,"OneClassSVM, KNN_Distance"
0,0185 106233,14,3,"IsolationForest, OneClassSVM, KNN_Distance"
14,2420 116077,10,1,LOF
33,8816 864475,9,2,"OneClassSVM, KNN_Distance"
18,4013 363390,9,1,EllipticEnvelope
24,6038 903499,8,2,"OneClassSVM, KNN_Distance"


In [5]:
SirenaDF = pd.read_csv("../resources/sirena.csv", sep="|")
aggregation_rules = {
    'FlightCode': 'mean',
    'SirenaBoardingMatch': 'mean',
    'SirenaFlightsMatch': 'mean',
    
    'LastName': 'first',
    'FirstName': 'first',
    'SecondName': 'first',
    
    'HasBirthDate': 'first',
    'PassengerAge': 'first',
    
    'Flight_DayOfWeek': 'mean',
    'Flight_Month': 'nunique',
    'DepartTimeMinutes': 'mean',
    
    'TrvCls_A': 'sum',
    'TrvCls_J': 'sum',
    'TrvCls_P': 'sum',
    'TrvCls_Y': 'sum'
}

passenger_df = SirenaDF.groupby('PaxName').agg(aggregation_rules).reset_index()

IDS = passenger_df['PaxName'].copy()
X = passenger_df.drop(columns=['PaxName'])

sp = spies_report(X, IDS)
sp.to_csv("sirena.csv", sep="|", index="false")
display(sp)

Поиск аномалий


/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(



----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
                           ID  Anomaly_Score  Rank
 DANILOVICH ALEKSEI PARAMONOV       0.119985     1
    DANILOVNA EMILIIA OZEROVA       0.103766     2
  ARSENEVICH GRIGORII KISELEV       0.102570     3
   ARSENEVNA IASMINA DAVYDOVA       0.101167     4
    LVOVICH ALEKSANDR KHROMOV       0.091188     5
 VLADISLAVOVNA KARINA NAUMOVA       0.089444     6
    DANILOVNA ADELINA SURKOVA       0.089418     7
     SAVVOVNA OLESIA MURAVEVA       0.089342     8
BOGDANOVNA NADEZHDA MERKULOVA       0.087635     9
   AIDAROVNA LIANA PLOTNIKOVA       0.085174    10
--------------------------------------------------

--- LOF ---
                           ID  Anomaly_Score  Rank
    GERMANOVNA ALENA CHERNYKH       2.121500     1
   ANDREEVICH GORDEI CHERNYKH       2.116583     2
  DANILOVNA VASILISA CHERNYKH       2.0

,ID,Total_Score,Models_Count,Models_List
25,PLATONOVNA KIRA DANILOVA,24,3,"LOF, OneClassSVM, KNN_Distance"
24,LVOVICH ALEKSANDR KHROMOV,23,3,"IsolationForest, OneClassSVM, KNN_Distance"
28,SEMENOVICH NAZAR SYCHEV,21,3,"LOF, OneClassSVM, KNN_Distance"
5,ARSENEVICH RADMIR GOLUBEV,20,3,"OneClassSVM, EllipticEnvelope, KNN_Distance"
17,DANILOVICH ALEKSEI PARAMONOV,20,3,"IsolationForest, OneClassSVM, KNN_Distance"
26,ROBERTOVNA ALLA SHEVTSOVA,20,3,"LOF, OneClassSVM, KNN_Distance"
7,ARSENEVNA IASMINA DAVYDOVA,15,3,"IsolationForest, EllipticEnvelope, KNN_Distance"
21,DANILOVNA VASILISA CHERNYKH,13,3,"LOF, OneClassSVM, KNN_Distance"
14,ARSENEVNA VERONIKA KHARITONOVA,10,1,EllipticEnvelope
23,GERMANOVNA ALENA CHERNYKH,10,1,LOF


In [6]:
Flights_Boarding_DF = pd.read_csv("../resources/flights_boarding.csv", sep="|")

if 'FlightTime' in Flights_Boarding_DF.columns and Flights_Boarding_DF['FlightTime'].dtype == object:
    ft = Flights_Boarding_DF['FlightTime'].astype(str)
    Flights_Boarding_DF['FlightTimeMinutes'] = (
        ft.str.slice(0, 2).astype(float) * 60 +
        ft.str.slice(3, 5).astype(float)
    )

aggregation_rules = {
    'Codeshare': 'mean',
    'Flight': 'mean',
    'PassengerFirstName': 'mean',
    'PassengerSecondName': 'mean',
    'PassengerLastName': 'mean',
    'PassengerSex': 'first',
    'BookingCode': 'mean',
    'TicketNumber': 'mean',
    'CodeShare': 'mean',
    'Flight_DayOfWeek': 'mean',
    'Flight_Month': 'nunique',
    'Airport': 'mean',
    'City': 'mean',
    'Country': 'mean',
    'BaggageTransit': 'sum',
    'BaggageDelayed': 'sum',
    'BaggageRegistered': 'sum'
}

passenger_df = Flights_Boarding_DF.groupby('PassengerDocument').agg(aggregation_rules).reset_index()

IDS = passenger_df['PassengerDocument'].copy()
X = passenger_df.drop(columns=['PassengerDocument'])

sp = spies_report(X, IDS)
sp.to_csv("flights_boarding.csv", sep="|", index=False)
display(sp)

Поиск аномалий


/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(



----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
         ID  Anomaly_Score  Rank
0016 277417       0.061421     1
5058 032032       0.055902     2
2908 898279       0.055135     3
1477 356912       0.054759     4
8483 273783       0.054420     5
5294 379207       0.052793     6
2211 426963       0.051903     7
2937 910860       0.051182     8
3381 195015       0.050705     9
2874 843094       0.049283    10
--------------------------------------------------

--- LOF ---
         ID  Anomaly_Score  Rank
6025 719450       2.983550     1
2468 127099       2.287839     2
8917 305885       2.285872     3
0903 193247       2.266384     4
9967 524999       2.235298     5
8048 395178       2.067222     6
9530 022510       2.049697     7
6561 078609       1.978905     8
8205 801009       1.925578     9
8007 894293       1.870297    10
---------------------------

,ID,Total_Score,Models_Count,Models_List
23,8007 894293,31,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
15,5058 032032,25,4,"IsolationForest, OneClassSVM, EllipticEnvelope..."
10,2908 898279,25,3,"IsolationForest, OneClassSVM, KNN_Distance"
6,2211 426963,19,4,"IsolationForest, OneClassSVM, EllipticEnvelope..."
1,0241 175441,19,3,"OneClassSVM, EllipticEnvelope, KNN_Distance"
21,6955 916298,12,2,"OneClassSVM, EllipticEnvelope"
12,3192 122949,11,2,"OneClassSVM, KNN_Distance"
0,0016 277417,10,1,IsolationForest
18,6025 719450,10,1,LOF
3,1181 213093,9,2,"OneClassSVM, KNN_Distance"


In [7]:
df = pd.read_csv("../resources/boarding_operational.csv", sep="|")

aggregation_rules = {
    'PassengerFirstName': 'mean',
    'PassengerSecondName': 'mean',
    'PassengerLastName': 'mean',
    'PassengerSex': 'first',
    'BookingCode': 'mean',
    'FlightTime': 'mean',
    'FlightNumber': 'mean',
    'CodeShare': 'mean',
    'boardFlightMatch': 'mean',
    'Fare': 'mean',
    'BaggageTransit': 'sum',
    'BaggageDelayed': 'sum',
    'BaggageRegistered': 'sum',
    'PassengerAge': 'first',
    'ClassY': 'sum',
    'ClassJ': 'sum',
    'ClassP': 'sum',
    'FF_Airline_SU': 'sum',
    'FF_Airline_KE': 'sum',
    'FF_Airline_DT': 'sum',
    'Airport': 'mean'
}

passenger_df = df.groupby('PassengerDocument').agg(aggregation_rules).reset_index()

IDS = passenger_df['PassengerDocument'].copy()
X = passenger_df.drop(columns=['PassengerDocument'])

sp = spies_report(X, IDS)
sp.to_csv("boarding_operational.csv", sep="|", index=False)
display(sp)

Поиск аномалий


/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:188: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-49.597773445909390 > -107.166608332999985). You may want to try with a higher value of support_fraction (current value: 0.503).
  warnings.warn(
/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:188: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-107.157849462181858 > -109.849616018034823). You may want to try with a higher value of support_fraction (current value: 0.503).
  warnings.warn(
/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:188: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-45.769814703404755 > -105.947505177965283). You may wan


----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
         ID  Anomaly_Score  Rank
2211 426963       0.073928     1
5294 379207       0.070831     2
1889 427649       0.070751     3
5058 032032       0.066261     4
1477 356912       0.066004     5
7043 974968       0.065728     6
1647 595790       0.065617     7
8421 352449       0.065472     8
3381 195015       0.065052     9
6695 351151       0.064812    10
--------------------------------------------------

--- LOF ---
         ID  Anomaly_Score  Rank
8007 894293       2.845137     1
8421 352449       2.052325     2
3505 707537       2.032051     3
0241 175441       1.859861     4
5879 481501       1.852779     5
3710 150996       1.820879     6
7767 309620       1.809874     7
2211 426963       1.783809     8
3941 182395       1.745868     9
4819 170172       1.741885    10
---------------------------

,ID,Total_Score,Models_Count,Models_List
23,8007 894293,40,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
25,8421 352449,31,5,"IsolationForest, LOF, OneClassSVM, EllipticEnv..."
7,2211 426963,30,5,"IsolationForest, LOF, OneClassSVM, EllipticEnv..."
1,0241 175441,27,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
11,3710 150996,27,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
13,4819 170172,16,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
6,1978 571330,9,1,EllipticEnvelope
16,5294 379207,9,1,IsolationForest
5,1889 427649,8,1,IsolationForest
10,3505 707537,8,1,LOF


In [8]:
df = pd.read_csv("../resources/pass_boarding.csv", sep="|")
aggregation_rules = {
    'Очередь_регистрации': 'mean',
    'PassengerFirstName': 'mean',
    'PassengerSecondName': 'mean',
    'PassengerLastName': 'mean',
    'PassengerSex': 'first',
    'BookingCode': 'mean',
    'FlightTime': 'mean',
    'FlightNumber': 'mean',
    'CodeShare': 'mean',
    'Destination': 'mean',
    'boardFlightMatch': 'mean',
    'OperatFlightMatch': 'mean',
    'PassengerAge': 'first',
    'PassportSeriesLeft': 'first',
    'PassportSeriesRight': 'first',
    'TicketNumber': 'sum',
    'BaggageTransit': 'sum',
    'BaggageDelayed': 'sum',
    'BaggageRegistered': 'sum',
    'Flight_DayOfWeek': 'mean',
    'Flight_Month': 'nunique',
    'ClassY': 'sum',
    'ClassJ': 'sum',
    'ClassP': 'sum',
    'City': 'mean',
    'Airport': 'mean',
    'BonusCard': 'max'
}

passenger_df = df.groupby('Document').agg(aggregation_rules).reset_index()

IDS = passenger_df['Document'].copy()
X = passenger_df.drop(columns=['Document'])

sp = spies_report(X, IDS)
sp.to_csv("pass_boarding.csv", sep="|", index=False)
display(sp)

Поиск аномалий


/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(



----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
         ID  Anomaly_Score  Rank
0389 510918       0.078760     1
5293 500602       0.067232     2
8560 808153       0.062200     3
5462 603688       0.060311     4
2294 770067       0.059387     5
6298 955090       0.059257     6
4592 146429       0.055290     7
4959 862296       0.052944     8
0977 338850       0.047693     9
9074 223766       0.047514    10
--------------------------------------------------

--- LOF ---
         ID  Anomaly_Score  Rank
8296 978071       1.629960     1
0548 488693       1.591156     2
9579 067482       1.554825     3
9777 886483       1.540321     4
5293 500602       1.527368     5
1770 182105       1.512739     6
6424 452421       1.500467     7
9679 942167       1.498753     8
7749 971945       1.495225     9
0974 927924       1.458141    10
---------------------------

,ID,Total_Score,Models_Count,Models_List
16,5293 500602,35,4,"IsolationForest, LOF, OneClassSVM, KNN_Distance"
7,1381 919028,27,3,"OneClassSVM, EllipticEnvelope, KNN_Distance"
29,9579 067482,25,3,"LOF, OneClassSVM, KNN_Distance"
2,0548 488693,19,3,"LOF, EllipticEnvelope, KNN_Distance"
26,8296 978071,19,3,"LOF, EllipticEnvelope, KNN_Distance"
20,6477 314549,14,2,"OneClassSVM, KNN_Distance"
18,6298 955090,13,3,"IsolationForest, OneClassSVM, KNN_Distance"
1,0389 510918,12,2,"IsolationForest, OneClassSVM"
23,7260 430301,10,2,"OneClassSVM, KNN_Distance"
13,4325 574940,9,1,EllipticEnvelope


In [9]:
df = pd.read_csv("../resources/loyalty_operational.csv", sep="|")

aggregation_rules = {
    'FirstName': 'mean',
    'LastName': 'mean',
    'Departure': 'mean',
    'Arrival': 'mean',
    'FF_Number': 'mean',
    'OperatFlightMatch': 'mean',
    'FlightOperationalMatch': 'mean',
    'ClassY': 'sum',
    'ClassJ': 'sum',
    'ClassP': 'sum',
    'BonusProgramA': 'sum',
    'BonusProgramD': 'sum',
    'BonusProgramK': 'sum',
    'Flight_DayOfWeek': 'mean',
    'Flight_Month': 'nunique',
    'Airport': 'mean',
    'FF_Airline_SU': 'sum',
    'FF_Airline_KE': 'sum',
    'FF_Airline_DT': 'sum'
}

passenger_df = df.groupby('Name').agg(aggregation_rules).reset_index()

IDS = passenger_df['Name'].copy()
X = passenger_df.drop(columns=['Name'])

sp = spies_report(X, IDS)
sp.to_csv("loyalty_operational.csv", sep="|", index=False)
display(sp)

Поиск аномалий

----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
                ID  Anomaly_Score  Rank
     ALLA RUSAKOVA       0.086187     1
     PETR KOCHETOV       0.084620     2
      VLAD GOLUBEV       0.080347     3
      OLEG ANTIPOV       0.073350     4
   DIANA SOFRONOVA       0.070824     5
   MILENA ZAYTSEVA       0.069915     6
    RADMIR KORNEEV       0.068503     7
      SAVVA MARKOV       0.068409     8
     MADINA LOSEVA       0.068329     9
ANZHELIKA BELIKOVA       0.065095    10
--------------------------------------------------

--- LOF ---
               ID  Anomaly_Score  Rank
TIMOFEY DRUZHININ       2.204499     1
  MARINA GORELOVA       2.155499     2
   VADIM PLATONOV       1.959899     3
    PETR KOCHETOV       1.923482     4
  ULIANA EROFEEVA       1.877531     5
    LILIYA ZOTOVA       1.809951     6
YASMINA ANISIMOVA       1

,ID,Total_Score,Models_Count,Models_List
16,PETR KOCHETOV,39,5,"IsolationForest, LOF, OneClassSVM, EllipticEnv..."
20,TIMOFEY DRUZHININ,39,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
11,MARINA GORELOVA,35,4,"LOF, OneClassSVM, EllipticEnvelope, KNN_Distance"
15,OLEG ANTIPOV,28,4,"IsolationForest, OneClassSVM, EllipticEnvelope..."
3,DIANA SOFRONOVA,20,4,"IsolationForest, OneClassSVM, EllipticEnvelope..."
1,ALLA RUSAKOVA,13,3,"IsolationForest, OneClassSVM, KNN_Distance"
7,IAROSLAVA SNEGIREVA,12,2,"OneClassSVM, KNN_Distance"
24,VLAD GOLUBEV,11,3,"IsolationForest, OneClassSVM, KNN_Distance"
18,SAVVA MARKOV,9,3,"IsolationForest, OneClassSVM, KNN_Distance"
19,TIKHON ZELENIN,8,2,"OneClassSVM, KNN_Distance"


In [10]:
df = pd.read_csv("../resources/loyalty_flights.csv", sep="|")

df = df.dropna(subset=['Name'])

aggregation_rules = {
    'FirstName': 'mean',
    'LastName': 'mean',
    'Departure': 'mean',
    'Arrival': 'mean',
    'Fare': 'mean',
    'LoyaltyOperationalMatch': 'mean',
    'Codeshare': 'mean',
    'boardFlightMatch': 'mean',
    'FlightOperationalMatch': 'mean',
    'SirenaFlightsMatch': 'mean',
    'BonusProgramA': 'sum',
    'BonusProgramD': 'sum',
    'BonusProgramK': 'sum',
    'BonusProgramFB': 'sum',
    'Flight_DayOfWeek': 'mean',
    'Flight_Month': 'nunique',
    'Airport': 'mean',
    'City': 'mean',
    'Country': 'mean'
}

passenger_df = df.groupby('Name').agg(aggregation_rules).reset_index()

IDS = passenger_df['Name'].copy()
X = passenger_df.drop(columns=['Name'])

X = X.fillna(0)

sp = spies_report(X, IDS)
sp.to_csv("loyalty_flights.csv", sep="|", index=False)
display(sp)

Поиск аномалий


/home/kirill/.local/share/pipx/venvs/jupyter/lib/python3.12/site-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(



----------------------------------------------------------------------
Подозрительные
----------------------------------------------------------------------

--- IsolationForest ---
                 ID  Anomaly_Score  Rank
IAROSLAVA SNEGIREVA       0.094009     1
      IGNAT DROZDOV       0.075733     2
      ALLA RUSAKOVA       0.071462     3
      PETR KOCHETOV       0.067970     4
  TIMOFEY DRUZHININ       0.067700     5
   DARINA BELYAKOVA       0.064895     6
      NIKOLAY ZIMIN       0.061566     7
    VALERII NIKITIN       0.060930     8
 VASILIY NIKOL'SKIY       0.060404     9
      DANIEL PANKOV       0.058546    10
--------------------------------------------------

--- LOF ---
                 ID  Anomaly_Score  Rank
  TIMOFEY DRUZHININ       2.644366     1
      PETR KOCHETOV       2.591677     2
STANISLAV VORONTSOV       2.534999     3
   KAMILLA CHERNYKH       2.313437     4
      AMINA OSIPOVA       2.303929     5
       DEMID SHUBIN       2.300780     6
    ARINA DOROF

,ID,Total_Score,Models_Count,Models_List
26,TIMOFEY DRUZHININ,26,3,"IsolationForest, LOF, OneClassSVM"
24,PETR KOCHETOV,25,3,"IsolationForest, LOF, OneClassSVM"
5,DARINA BELYAKOVA,21,4,"IsolationForest, OneClassSVM, EllipticEnvelope..."
17,MATVEI VESHNIAKOV,17,3,"OneClassSVM, EllipticEnvelope, KNN_Distance"
3,ARINA DOROFEEVA,14,2,"LOF, KNN_Distance"
33,ZAKHAR VOLKOV,13,2,"OneClassSVM, KNN_Distance"
29,VALERIYA MEDVEDEVA,10,2,"LOF, KNN_Distance"
13,IAROSLAVA SNEGIREVA,10,1,IsolationForest
34,ZARINA MARKELOVA,10,1,EllipticEnvelope
6,DARINA POZDNYAKOVA,9,1,EllipticEnvelope


In [11]:
df = pd.read_csv("SPIES.csv", sep="|")

aggregation_rules = {
    'Score': 'sum',
    'ModelsCount': 'sum',
    'File': lambda x: ', '.join(x.astype(str))
}

df = df.groupby('Data').agg(aggregation_rules).reset_index()
df = df.sort_values(by='Score', ascending=False).reset_index(drop=True)

df.to_csv("SPIES_ready.csv", sep="|", index="false")
df

,Data,Score,ModelsCount,File
0,8007 894293,71,8,"flights_boarding.csv , boarding_operational...."
1,TIMOFEY DRUZHININ,65,7,"loyality_operational.csv, loyality_flights.csv"
2,PETR KOCHETOV,64,8,"loyality_operational.csv, loyality_flights.csv"
3,DANILOVNA VASILISA CHERNYKH,55,8,"boarding.csv , sirena.csv"
4,2211 426963,49,9,"flights_boarding.csv , boarding_operational...."
5,0241 175441,46,7,"flights_boarding.csv , boarding_operational...."
6,MARINA GORELOVA,35,4,loyality_operational.csv
7,5293 500602,35,4,pass_boarding.csv
8,8421 352449,31,5,boarding_operational.csv
9,OLEG ANTIPOV,28,4,loyality_operational.csv
